In [ ]:
# Cell 1 — Imports and reconnect to existing ChromaDB that was set up in notebook 2
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_ollama import OllamaLLM

# Use the absolute path to the ChromaDB database.
# The relative path "data/chroma_db" resolves differently depending on which
# directory the notebook is opened from. The absolute path is unambiguous.
CHROMA_PATH = r"c:\Users\pjay4\Documents\My_ML_Project\rag-project\data\chroma_db"

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(name="nppf")

print(f"Collection loaded: {collection.count()} chunks")

In [ ]:
# Cell 2 — Retrieval function
def retrieve(query: str, k: int = 5) -> list[dict]:
    """
    Embed `query` and return the top-k most similar chunks from ChromaDB.
    
    Returns a list of dicts, each with:
      - 'text':      the chunk text
      - 'para_num':  the NPPF paragraph number (or None if not set)
      - 'chunk_id':  the chunk identifier
      - 'distance':  cosine distance from query (lower = more similar)
    """
    # Embed the query — produces a (384,) vector
    query_vector = embed_model.encode(query).tolist()  # ChromaDB expects a plain list
    
    # Query ChromaDB — returns top-k chunks by cosine similarity
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    # Unpack the results — ChromaDB nests results in lists because you can
    # submit multiple queries at once; we submitted one, so index [0]
    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunks.append({
            "text": doc,
            "para_num": meta.get("para_num"),
            "chunk_id": meta.get("chunk_id"),
            "distance": round(dist, 4)
        })
    
    return chunks


# Quick smoke test — run one query to verify retrieval is working
test_chunks = retrieve("What is the presumption in favour of sustainable development?", k=5)
for i, c in enumerate(test_chunks):
    print(f"\n--- Chunk {i+1} | para_num={c['para_num']} | distance={c['distance']} ---")
    print(c['text'][:300])

In [ ]:
# Cell 3 — Prompt construction

def build_prompt(query: str, chunks: list[dict]) -> str:
    """
    Construct the augmented prompt for the LLM.
    
    Structure:
      [SYSTEM]   Role + citation rules
      [CONTEXT]  Retrieved chunks, each labelled with para_num
      [QUESTION] The user query
    """
    system = (
        "You are a planning policy assistant with expert knowledge of the "
        "National Planning Policy Framework (NPPF). "
        "Answer the question below using ONLY the policy excerpts provided. "
        "For every claim you make, cite the paragraph number in the format [para X]. "
        "If the excerpts do not contain sufficient information to answer the question, "
        "respond with: 'I could not find relevant policy guidance in the retrieved sections.'"
    )
    
    # Format each chunk — label with paragraph number where available
    context_parts = []
    for i, chunk in enumerate(chunks):
        label = f"para {chunk['para_num']}" if chunk['para_num'] else f"chunk {chunk['chunk_id']}"
        context_parts.append(f"[{label}]\n{chunk['text']}")
    
    context_block = "\n\n".join(context_parts)
    
    prompt = (
        f"{system}\n\n"
        f"POLICY EXCERPTS:\n{context_block}\n\n"
        f"QUESTION: {query}\n\n"
        f"ANSWER:"
    )
    
    return prompt


# Instantiate the Ollama LLM via langchain-ollama
# base_url defaults to http://localhost:11434 — Ollama must be running
llm = OllamaLLM(model="llama3.2:3b", temperature=0.1)
# Low temperature (0.1) because we want factual, grounded answers —
# not creative variation. Higher temperature increases the risk of
# hallucination in a retrieval context.

print("LLM instantiated. Model: llama3.2")

In [ ]:
# Cell 4 — Full RAG pipeline function

def rag_query(query: str, k: int = 5) -> dict:
    """
    Full RAG pipeline: query → retrieve → generate.
    
    Returns a dict with:
      - 'query':    the original question
      - 'answer':   the LLM-generated answer
      - 'chunks':   the retrieved chunks (text, para_num, distance)
    """
    chunks = retrieve(query, k=k)
    prompt = build_prompt(query, chunks)
    answer = llm.invoke(prompt)
    
    return {
        "query": query,
        "answer": answer,
        "chunks": chunks
    }


def print_result(result: dict) -> None:
    """Pretty-print a RAG result for notebook inspection."""
    print("=" * 70)
    print(f"QUERY: {result['query']}")
    print("-" * 70)
    print(f"ANSWER:\n{result['answer']}")
    print("-" * 70)
    print("RETRIEVED CHUNKS:")
    for c in result['chunks']:
        print(f"  para_num={c['para_num']} | distance={c['distance']} | {c['text'][:120]}...")
    print("=" * 70)


# End-to-end test on one question before running the full set
result = rag_query("What is the presumption in favour of sustainable development?")
print_result(result)

In [ ]:
# Cell 5 — 10 test questions spanning the NPPF

test_questions = [
    # Sustainable development (Chapter 2)
    "What is the presumption in favour of sustainable development?",
    
    # Housing (Chapter 5)
    "How should local planning authorities calculate their housing need?",
    
    # Green Belt (Chapter 13)
    "What are the five purposes of the Green Belt?",
    
    # Design (Chapter 12)
    "What principles should guide good design in new developments?",
    
    # Transport (Chapter 9)
    "What should transport assessments consider for new developments?",
    
    # Climate change (Chapter 14)
    "How should planning policy support climate change adaptation?",
    
    # Town centres (Chapter 7)
    "What is the sequential test for retail and town centre development?",
    
    # Natural environment (Chapter 15)
    "What protections apply to ancient woodland and veteran trees?",
    
    # Viability (Chapter 5 / general)
    "When can a developer argue that affordable housing requirements affect viability?",
    
    # Decision-making (Chapter 4)
    "What weight should be given to policies in an out-of-date local plan?",
]

print(f"{len(test_questions)} test questions ready.")

In [ ]:
# Cell 6 — Run all 10 questions through the pipeline

# This will take a few minutes — Ollama runs locally on CPU
# Each question triggers one embedding call + one ChromaDB query + one LLM call

results = []
for i, q in enumerate(test_questions):
    print(f"Running question {i+1}/10: {q[:60]}...")
    r = rag_query(q, k=5)
    results.append(r)
    print(f"  Done. Answer length: {len(r['answer'])} chars")

print("\nAll 10 questions complete.")

In [ ]:
# Cell 7 — Print all results for manual review

for r in results:
    print_result(r)
    print()

## Evaluation — 10 Test Questions (Session 3)

**Evaluation criteria (manual):**
- **Grounded**: Is the answer supported by the retrieved chunks? (Y/N)
- **Citations correct**: Does the answer cite paragraph numbers that match the retrieved chunks? (Y/N)
- **Complete**: Does the answer address the question adequately? (Y/Partial/N)

| # | Question (abbreviated) | Grounded | Citations correct | Complete | Notes |
|---|------------------------|----------|-------------------|----------|-------|
| 1 | Presumption in favour of SD |Y |Y |N |Does not answer the question - but might be too complex for the Llama3.2:3b model; answer is literal quote from text |
| 2 | Housing need calculation |Y |Y |Y | |
| 3 | Five purposes of Green Belt |Y |Y |Y | |
| 4 | Good design principles |Y |Y |Y |Uses multiple paras to formulate the answer, citing each of them in turn |
| 5 | Transport assessment |Y |Y |Y | |
| 6 | Climate change adaptation |Y |Y |Y |Uses two paras for answer but also cites another as a note at the end |
| 7 | Sequential test (retail) |Y |Y |Partial |Second para cited is from a different context and so that part of the answer is not correct|
| 8 | Ancient woodland / veteran trees |Y |Y |Y |Citiation comes from part (c) of a bigger paragraph, so retrived chnuk only has "para c" rather than 193 |
| 9 | Viability and affordable housing |Y |Y |Partial |Not enough information within the chunks |
| 10 | Out-of-date local plan weight |Y |Y |Y | |

**Overall observations:**
- Retrieval quality (distance scores, relevance of top-k chunks): Good overall - for well-define policy questions distances are low (0.2 - 0.3) and for more complex questions a little higher; closer matches produced more complete answers. Two failure modes observed: (1) semantic collision where the same phrase ("sequential test") appears in different policy contexts, pulling irrelevant chunks into the top-k; (2) para_num=-1 metadata for glossary/sub-paragraph chunks, which are retrieved correctly but cited imprecisely.
- Citation accuracy: Very good - all answers had relevant paragraphs cited
- Failure modes observed (hallucination, wrong paragraph cited, incomplete answer): Errors seem to come from the questions not being answers by the chunks retrieved, or being slightly too complex
- Best-performing question type: Direct questions that can be answered quite simply - probably because they support the retrieval by giving the best words to match up with the chunks
- Worst-performing question type: Policy questions which cannot be answered in a single paragraph

In [ ]:
# Cell 8a — Set API key (run this cell once, do not save the key in the notebook)

import os
os.environ["ANTHROPIC_API_KEY"] = "sk-"  # paste key here, run, then delete
print("API key set.")


In [ ]:
# Cell 8b — Claude API comparison on the same 10 test questions
# Uses identical retrieval (same chunks) — only generation differs.
# This isolates whether failures are retrieval problems or generation problems.

import anthropic

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

def rag_query_claude(query: str, k: int = 5) -> dict:
    """
    Same RAG pipeline as rag_query(), but calls Claude instead of Ollama.
    Retrieval is identical — only the generation step changes.
    """
    chunks = retrieve(query, k=k)       # same retrieval function as before
    prompt = build_prompt(query, chunks) # same prompt construction as before

    message = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "query": query,
        "answer": message.content[0].text,
        "chunks": chunks  # same chunks as Ollama would have received
    }


# Run all 10 questions — this will make 10 API calls
claude_results = []
for i, q in enumerate(test_questions):
    print(f"Running question {i+1}/10: {q[:60]}...")
    r = rag_query_claude(q, k=5)
    claude_results.append(r)

print("\nAll 10 questions complete.")

In [ ]:
# Cell 8c — Side-by-side comparison: Ollama vs Claude
# For each question, prints both answers so you can compare directly.
# The retrieved chunks are identical — any difference is pure generation quality.

for i, (ollama_r, claude_r) in enumerate(zip(results, claude_results)):
    print("=" * 70)
    print(f"Q{i+1}: {ollama_r['query']}")
    print()
    print("--- OLLAMA (llama3.2:3b) ---")
    print(ollama_r['answer'])
    print()
    print("--- CLAUDE (sonnet-4-5) ---")
    print(claude_r['answer'])
    print()

It seems like the claude LLM does give richer answers and is able to reason better based on the retrieved chunks (especially Q1 and Q9) as well as removing irrelevant paras (Q7). But generally it seems like the issues arise from the retrieval step, rather than the LLM - unless it is quite a complex question where it is difficult to fully answer based on the chunks, or slightly irrelevant chunks are retrieved.

## Model Comparison: Ollama (llama3.2:3b) vs Claude (Sonnet 4.5)

**Key finding:** Generation quality matters less than retrieval quality for straightforward policy questions. Both models answered Q2, Q3, Q5, Q6 and Q10 correctly with the same retrieved chunks. The performance gap widened on:

- **Complex questions** (Q4, Q6): Claude structured multi-paragraph answers more coherently, synthesising across chunks rather than listing them sequentially.
- **Retrieval failures** (Q1, Q9): Claude correctly flagged missing information; llama3.2:3b produced confident but misleading answers. This is a meaningful safety difference for a planning policy tool — a practitioner acting on llama's Q9 answer would be misinformed.
- **Semantic collisions** (Q7): Claude filtered out the irrelevant flood risk sequential test chunk; llama incorporated it into its answer incorrectly.

**Implication for next steps:** Re-ranking (cross-encoder) is the highest-value improvement — it addresses the root cause of failures Q1, Q7 and Q9. 
Swapping the generation model is a secondary concern; the local llama3.2:3b model is adequate for well-retrieved questions and has the major advantages of zero API cost and no data leaving the local machine (relevant for a 
planning tool that could eventually handle sensitive development proposals).